# LightOnOCR-2-1B + QLoRA for **Sinhala Handwritten** OCR

Second architecture track. Fine-tunes **`lightonai/LightOnOCR-2-1B`** on
`Datasets/handwritten-data` (908 train / 227 test), warm-started from the authors' own
Sinhala **print** adapter.

**Goal: beat CER 0.5253** (TrOCR printed→handwritten, IJDAR Table 5) — the current state of
the art for handwritten Sinhala, measured on this exact 227-image test split.

> The DeepSeek-OCR track reached **0.7059**. See `../GuideForLightOnOCR/README.md` for why this
> architecture is expected to do better, and `../GuideForDeepSeek/learnings.md` for the
> diagnosis that motivated it.

---

## Why LightOnOCR, specifically

The DeepSeek run failed for three measurable reasons. Native-resolution encoding attacks all three:

| DeepSeek-OCR run's problem | Measured evidence | LightOnOCR |
|---|---|---|
| Aspect distortion | up to **2.42×** vertical stretch from `dynamic_preprocess` | **Fixed** — Pixtral keeps aspect, no tiling |
| Token waste → training cut short | ~1156 tokens for a 3-char word; stopped at **4.49/8** epochs; VRAM 4.89/14.6 GB | **Fixed** — ~47 tokens for a median crop |
| Overfitting | train loss 1.04→0.40 while val CER flat; **86M** trainable on 817 samples | **Reduced** — no MoE; their adapter is 140 MB vs DeepSeek's 281 MB |
| LM-prior hallucination (`කොළඹ`→`ගමදීම`) | real Sinhala words, wrong ones | **Reduced** — Qwen3-1B is a weaker prior than a 3B MoE |

Sinhala diacritics are small. Stretching a 21:1 strip by 2.42× vertically destroys exactly the
stroke detail that separates characters, so the decoder falls back on guessing plausible words —
the identical failure the IJDAR paper documented for TrOCR + SinBERT
(*"predicts incorrect text in an attempt to make the output a cohesive phrase"*).

## The warm start is what makes this viable

LightOnOCR has **no Sinhala** in its language list (en, fr, de, es, it, nl, pt, sv, da, zh, ja),
and scored **CER 0.8805 zero-shot** on Sinhala print — worse than DeepSeek-OCR V1's 0.6146.
Learning Sinhala script from our 17,122 training characters alone would be hopeless.

But **`avishadilhara/sinhala-lightonocr-2-1b-Qlora` is public**, and it already learned Sinhala
from 707 dense Act pages (~1–2M characters) to reach **CER 0.0105**. Its `adapter_config.json`:
`r=32, lora_alpha=64, lora_dropout=0.1`, base `lightonai/LightOnOCR-2-1B` — exactly the paper's
**Experiment 7**. This notebook continues training *that* adapter on handwriting, which is the
printed→handwritten two-stage recipe that took TrOCR from 0.9940 to 0.5253.

The equivalent warm start failed in the DeepSeek track (the print adapter scored **5.37**,
worse than the untouched base at 5.07) because it was trained on *pages* and DeepSeek forces
every input into a square canvas. Here the granularity mismatch should matter far less: a native
resolution encoder processes a line crop and a page with the same mechanism, just fewer tokens.
**Cell 8 tests this directly, before any training.**

## Faithful to the authors' Experiment 7

Hyperparameters, collator and chat format come from their own repo
(`Cross-Temporal-Sinhala-OCR-Avisha-Dilhara/Model-finetuning/LightOnOCR-2-1B OCR/…(EX07).ipynb`),
not from guesswork. Deviations are deliberate and marked `DEVIATION` in the code:

| | Their Exp 7 | Here | Why |
|---|---|---|---|
| GPU / dtype | RTX 4090, **bf16** | Kaggle T4, **fp16** | T4 is `sm_75`, no native bf16 |
| Epochs | 20 | 20 with early stopping | 817 samples overfit fast |
| Best-model metric | `eval_loss` | **validation CER** | loss and CER decouple; we report CER |
| `max_new_tokens` | 2048 | derived from data (~146) | our targets are ≤65 tokens |
| Test samples | 50 (in their FT notebook) | all 227 | full split |
| Warm start | none | their Sinhala print adapter | stage 1 we cannot otherwise afford |

## Running this on Kaggle

- **Accelerator: `GPU T4 ×2`.** Not P100 (`sm_60` is unsupported by Kaggle's torch — proven), not TPU.
- **Internet: ON.** **Input:** attach `handwritten-data2`. No `HF_TOKEN` needed — all repos public.
- **This notebook needs `transformers==5.0.0`** (LightOnOCR's classes live there). The DeepSeek
  notebook needs `4.56.2` and they are incompatible — never run both in one Kaggle session.


In [6]:
# ============================================================================
# Cell 1 — Configuration and environment
# ============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# LightOnOCR's classes (LightOnOcrProcessor / LightOnOcrForConditionalGeneration) ship in
# transformers 5.x. This is the version the authors' Exp 7 notebook pins.
# NOTE: incompatible with the DeepSeek notebook, which needs 4.56.2. Separate sessions.
!pip install -q -U datasets accelerate peft
!pip install -q transformers==5.0.0
!pip install -q "huggingface-hub>=1.3.1"
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install -q jiwer "Pillow>=11.3.0"

import sys, random, json, math, time, gc, csv, contextlib, inspect
import numpy as np
import torch

# Verify the pins actually landed. `-q` hides pip's resolver errors, and a wrong
# transformers version would only surface in Cell 4 after Cells 2-3 walked 1135 images.
import transformers, peft, bitsandbytes, huggingface_hub
print(f"transformers {transformers.__version__} | peft {peft.__version__} | "
      f"bitsandbytes {bitsandbytes.__version__} | hub {huggingface_hub.__version__}")
assert transformers.__version__.startswith("5.0"), (
    f"need transformers==5.0.0, got {transformers.__version__}. Re-run this cell, then "
    f"Kaggle -> Run -> Restart session, then Run All.")
from transformers import LightOnOcrProcessor, LightOnOcrForConditionalGeneration  # fail HERE, not in Cell 4
print("LightOnOcr classes import OK")

NOTEBOOK_VERSION = "2026-09-14.1"
print("=" * 70)
print(f"lightOnOcrForHandWrittenText.ipynb   revision {NOTEBOOK_VERSION}")
print("=" * 70)

# ---------------------------------------------------------------- CONFIG ----
CFG = dict(
    smoke_test           = False,   # tiny end-to-end pass to validate the pipeline
    run_zero_shot        = True,    # the decisive probe: does their print adapter transfer?
    do_train             = True,
    run_final_test_eval  = True,

    # --- model ---
    base_repo            = "lightonai/LightOnOCR-2-1B",
    print_adapter_repo   = "avishadilhara/sinhala-lightonocr-2-1b-Qlora",
    warm_start           = True,
    load_in_4bit         = True,
    lora_r               = 32,      # must be 32 when warm_start=True (their adapter's shapes)
    lora_alpha           = 64,
    lora_dropout         = 0.1,

    # --- input representation (their Exp 7) ---
    longest_edge         = 1540,
    max_length           = 4096,
    # Per-image cap on vision patches, applied BEFORE the processor. Backward OOM'd at
    # 3,190 patches: the Pixtral tower's 24 attention matrices (16 heads, bf16) come to
    # 24 x 311 MB = 7.28 GB, which matches the 7.21 GiB allocation that failed -- i.e. its
    # activations are NOT being gradient-checkpointed. At 2,400 the same worst case is
    # 4.12 GB and only 14 of 1135 images (1.2%) are downscaled at all, so `longest_edge`
    # stays at their Exp 7 value of 1540 for 98.8% of the data.
    max_patches_per_image = 2400,

    # --- data ---
    val_fraction         = 0.10,
    augment              = True,

    # --- optimisation (their Exp 7, except where noted) ---
    epochs               = 20,
    # Their Exp 7 used batch 4 x accum 1. A real T4 run OOM'd there, because Pixtral
    # CONCATENATES patches across a batch and pads every image to the batch's max height:
    # the 4 largest crops became 4 x 3190 = 12,760 patches and one attention matrix alone
    # needed ~7.2 GB. batch 1 x accum 4 keeps the EFFECTIVE batch at 4 (mathematically the
    # same gradient, no batchnorm here) while peak memory becomes the single largest crop
    # (3,190 patches, ~0.3 GB) with zero padding waste.
    per_device_batch     = 1,
    grad_accum           = 4,
    # Their Exp 7 lr=2e-4 is a COLD-START rate. We continue an adapter that already
    # converged to CER 0.0105, so a cold-start rate over ~4000 steps would very likely wipe
    # the printed-Sinhala representation (standard catastrophic forgetting). DEVIATION:
    # use a stage-2 rate when warm-starting.
    lr                   = 2e-4,    # used only when warm_start=False
    lr_warm_start        = 2e-5,    # used when warm_start=True
    lr_scheduler         = "linear",
    warmup_steps         = 10,
    weight_decay         = 0.001,
    max_grad_norm        = 1.0,
    optim                = "adamw_8bit",   # DEVIATION: 8-bit to fit a 16 GB T4
    grad_checkpointing   = True,

    # --- validation / early stopping (DEVIATION: CER, not eval_loss) ---
    # ~204 steps/epoch at batch 4. eval_every_steps=50 would mean ~81 validation passes
    # x 64 generations = 5184 in-training generations, several hours on a T4.
    # With batch 1 an "optimizer step" is 4 samples, so 817 samples ~= 204 optimizer steps
    # per epoch. 400 optimizer steps is about one eval every two epochs.
    eval_every_steps     = 400,
    val_eval_n           = 32,
    early_stop_patience  = 4,

    # --- budget ---
    train_hours_budget   = 6.5,
    eval_max_new_tokens  = 0,       # 0 => derive from measured target lengths
    zeroshot_n           = 227,     # probe (b): the print adapter -- the number that matters
    zeroshot_base_n      = 64,      # probe (a): the base model only needs to show it is hopeless
    zeroshot_max_new_tok = 192,
    seed                 = 3407,    # their seed
    assert_dataset_identity = True,

    # --- numeric precision ------------------------------------------------
    # "bf16" | "fp16" | "fp32" | "auto"(by compute capability).
    # A real T4 run with fp16 produced `loss = nan` on the FIRST batch. This model is
    # bfloat16 at every level of its config (top-level, text_config, vision_config) and
    # their Exp 7 ran `fp16=False, bf16=True`. Pixtral and Qwen3 are both well known to
    # overflow fp16's ~65504 ceiling. bf16 has fp32's exponent range, so it does not
    # overflow; on a T4 it is EMULATED (slower) but numerically correct.
    # If Cell 1's matmul probe fails under bf16 on this GPU, fall back to "fp32".
    force_dtype          = "bf16",
)
if CFG["smoke_test"]:
    CFG.update(epochs=1, zeroshot_n=6, zeroshot_base_n=6, val_eval_n=6,
               eval_every_steps=5, train_hours_budget=0.5)
    print(">>> SMOKE TEST MODE: results are NOT meaningful\n")

IN_KAGGLE  = os.path.isdir("/kaggle")
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else "."
RUN_DIR    = os.path.join(OUTPUT_DIR, "lightonocr_hw_run")
BEST_DIR   = os.path.join(RUN_DIR, "best_adapter")
os.makedirs(RUN_DIR, exist_ok=True)

# ------------------------------------------------------- GPU: fail fast -----
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Kaggle -> Settings -> Accelerator -> 'GPU T4 x2'. Not TPU.")

cap = torch.cuda.get_device_capability(0)
HAS_NATIVE_BF16 = cap[0] >= 8
_DT = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}
COMPUTE_DTYPE = (_DT[CFG["force_dtype"]] if CFG["force_dtype"] in _DT
                 else (torch.bfloat16 if HAS_NATIVE_BF16 else torch.float16))
USE_FP16_AMP  = (COMPUTE_DTYPE == torch.float16)

print(f"GPU              : {torch.cuda.get_device_name(0)}  (compute capability {cap[0]}.{cap[1]})")
print(f"torch / CUDA     : {torch.__version__} / {torch.version.cuda}")
print(f"total VRAM       : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"is_bf16_supported: {torch.cuda.is_bf16_supported()}   <-- True via EMULATION on T4")
print(f"native bf16      : {HAS_NATIVE_BF16}")
print(f"force_dtype      : {CFG['force_dtype']!r}")
print(f"=> COMPUTE_DTYPE : {COMPUTE_DTYPE}   (AMP fp16={USE_FP16_AMP})")
if COMPUTE_DTYPE == torch.bfloat16 and not HAS_NATIVE_BF16:
    print("   bf16 on a GPU without native bf16: EMULATED, so slower but numerically safe.")
    print("   Chosen deliberately: this model is bf16 at every config level and fp16 NaN'd")
    print("   on the first batch in a real T4 run (see GuideForLightOnOCR/learnings.md L3).")
elif COMPUTE_DTYPE == torch.float16:
    print("   WARNING: fp16 produced `loss = nan` on this model in a real T4 run.")
    print("   Set CFG['force_dtype']='bf16' unless you are deliberately re-testing that.")

# The installed torch must have kernels for THIS GPU. A P100 (sm_60) passes
# torch.cuda.is_available() and then dies mid-model-load. Fail here instead.
ARCH_LIST, DEV_ARCH = torch.cuda.get_arch_list(), f"sm_{cap[0]}{cap[1]}"
print(f"\ntorch built for  : {ARCH_LIST}")
print(f"this GPU needs   : {DEV_ARCH}")
if DEV_ARCH not in ARCH_LIST:
    raise RuntimeError(
        f"This PyTorch ({torch.__version__}) has NO kernels for {DEV_ARCH} "
        f"({torch.cuda.get_device_name(0)}); it supports {ARCH_LIST}.\n"
        f"FIX: Kaggle -> Settings -> Accelerator -> 'GPU T4 x2' (sm_75). Never P100 (sm_60)."
    )
try:
    _p = torch.randn(64, 64, device="cuda", dtype=COMPUTE_DTYPE)
    _ = (_p @ _p).sum().item(); del _p; torch.cuda.synchronize()
    print(f"CUDA smoke test  : OK ({COMPUTE_DTYPE} matmul ran on device)")
except Exception as e:
    raise RuntimeError(
        f"A trivial {COMPUTE_DTYPE} matmul failed on {torch.cuda.get_device_name(0)}: {e}\n"
        f"If the accelerator is already 'GPU T4 x2', this GPU cannot run {COMPUTE_DTYPE} "
        f"matmuls; set CFG['force_dtype']='fp32' (slower, more VRAM, but numerically safest) "
        f"and re-run. Do NOT fall back to 'fp16' -- it NaN'd on this model.")

# PEFT raises (not returns False) if torchao is installed but < 0.16.0. Kaggle ships 0.10.0,
# which killed get_peft_model() twice in the DeepSeek track. Neutralise the probe: we
# quantise with bitsandbytes, so torchao is irrelevant here.
import importlib
_patched = []
for _n, _m in list(sys.modules.items()):
    if _n.startswith("peft") and _m is not None and hasattr(_m, "is_torchao_available"):
        _f = getattr(_m, "is_torchao_available")
        if hasattr(_f, "cache_clear"):
            try: _f.cache_clear()
            except Exception: pass
        setattr(_m, "is_torchao_available", lambda: False); _patched.append(_n)
print(f"\nPEFT torchao probe disabled in {len(_patched)} already-loaded module(s): {_patched}")
print("   (Cell 4 re-applies this after peft is imported — that is the one that counts.)")

SEED = CFG["seed"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")

print(f"\nrun dir: {RUN_DIR}")
print(json.dumps({k: (v if isinstance(v,(int,float,bool,str,type(None))) else str(v))
                  for k,v in CFG.items()}, indent=2))


transformers 5.0.0 | peft 0.20.0 | bitsandbytes 0.50.2 | hub 1.11.0
LightOnOcr classes import OK
lightOnOcrForHandWrittenText.ipynb   revision 2026-09-14.1
GPU              : Tesla T4  (compute capability 7.5)
torch / CUDA     : 2.10.0+cu128 / 12.8
total VRAM       : 14.6 GB
is_bf16_supported: True   <-- True via EMULATION on T4
native bf16      : False
force_dtype      : 'bf16'
=> COMPUTE_DTYPE : torch.bfloat16   (AMP fp16=False)
   bf16 on a GPU without native bf16: EMULATED, so slower but numerically safe.
   Chosen deliberately: this model is bf16 at every config level and fp16 NaN'd
   on the first batch in a real T4 run (see GuideForLightOnOCR/learnings.md L3).

torch built for  : ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
this GPU needs   : sm_75
CUDA smoke test  : OK (torch.bfloat16 matmul ran on device)

PEFT torchao probe disabled in 4 already-loaded module(s): ['peft.import_utils', 'peft.utils.quantization_utils', 'peft.tuners.lora.torchao', 'peft.tune

In [7]:
# ============================================================================
# Cell 2 — Find the dataset by STRUCTURE, verify it is SinOCR-Handwritten
# ============================================================================
import os, csv

# Hard floor for the 2x2 patch merger is 15px on the short side; 32 leaves room for the
# augmentation rescale (x0.93) and for rounding. Enforced in Cell 5 and Cell 7.
MIN_SHORT_SIDE = 32

REQUIRED   = [("train","data.csv"),("train","images"),("test","data.csv"),("test","images")]
KNOWN_HINT = "/kaggle/input/datasets/danushamsc25/handwritten-data2"

def find_dataset_root(roots, max_depth=6):
    found=[]
    for root in roots:
        if not os.path.isdir(root): continue
        for dirpath, dirnames, _ in os.walk(root):
            if dirpath[len(root):].count(os.sep) > max_depth:
                dirnames[:] = []; continue
            if all(os.path.exists(os.path.join(dirpath,a,b)) for a,b in REQUIRED):
                found.append(dirpath); dirnames[:] = []
    return list(dict.fromkeys(found))

def print_tree(root, max_depth=4):
    if not os.path.isdir(root): print(f"(no such directory: {root})"); return
    for dirpath, dirnames, filenames in os.walk(root):
        d = dirpath[len(root):].count(os.sep)
        if d > max_depth: dirnames[:] = []; continue
        print("  "*d + (os.path.basename(dirpath) or dirpath) + "/")
        if d == max_depth:
            for fn in sorted(filenames)[:15]: print("  "*(d+1) + fn)

cands = find_dataset_root([KNOWN_HINT, "/kaggle/input", ".", "/kaggle/working"])
if not cands:
    print("Could not find the dataset (need train/ and test/, each with data.csv + images/).")
    print("\nTree of /kaggle/input:"); print_tree("/kaggle/input")
    raise FileNotFoundError("Attach the `handwritten-data2` dataset (Add Input -> Datasets).")

DATASET_ROOT = cands[0]
print(f"dataset root: {DATASET_ROOT}" + (f"   (others: {cands[1:]})" if len(cands)>1 else ""))

def load_split(root, split):
    samples, missing = [], []
    with open(os.path.join(root,split,"data.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            p = os.path.join(root, split, "images", f"{r['file_name']}.png")
            if os.path.exists(p): samples.append({"id":r["file_name"],"image":p,"text":r["text"]})
            else: missing.append(p)
    return samples, missing

TRAIN_ALL, m1 = load_split(DATASET_ROOT, "train")
TEST,      m2 = load_split(DATASET_ROOT, "test")
if m1 or m2:
    raise FileNotFoundError(f"Missing images: {len(m1)} train, {len(m2)} test. e.g. {(m1+m2)[:3]}")

# This assertion is what makes CER comparable to IJDAR's 0.5253.
IJDAR_TABLE2 = {"train":{"n":908,"chars":17122}, "test":{"n":227,"chars":4029}}
print("\nIntegrity check vs IJDAR Table 2 (SinOCR-Handwritten):")
ok_all = True
for name, rows in (("train",TRAIN_ALL), ("test",TEST)):
    n, chars = len(rows), sum(len(r["text"]) for r in rows)
    exp = IJDAR_TABLE2[name]; ok = (n==exp["n"] and chars==exp["chars"]); ok_all &= ok
    print(f"  {name:5}: images {n:5} (paper {exp['n']:5})   code points {chars:6} "
          f"(paper {exp['chars']:6})   {'MATCH' if ok else 'MISMATCH'}")
if ok_all:
    print("  => This IS the published SinOCR-Handwritten split; CER is comparable to 0.5253.")
elif CFG["assert_dataset_identity"]:
    raise AssertionError("Dataset does not match IJDAR Table 2; the comparison would be invalid.")

from PIL import Image
def stats(rows, label):
    q = lambda v,p: sorted(v)[min(len(v)-1,int(p*len(v)))]
    L=[len(r["text"]) for r in rows]
    sizes=[Image.open(r["image"]).size for r in rows]
    W=[s[0] for s in sizes]; H=[s[1] for s in sizes]; A=[s[0]/s[1] for s in sizes]
    print(f"\n{label} (n={len(rows)})")
    print(f"  chars/sample : min {min(L)}  med {q(L,.5)}  p95 {q(L,.95)}  max {max(L)}")
    print(f"  width  px    : min {min(W)}  med {q(W,.5)}  p95 {q(W,.95)}  max {max(W)}")
    print(f"  height px    : min {min(H)}  med {q(H,.5)}  p95 {q(H,.95)}  max {max(H)}")
    print(f"  aspect ratio : min {min(A):.1f}  med {q(A,.5):.1f}  max {max(A):.1f}")
    # Pixtral: longest edge scaled to <=LONGEST_EDGE, aspect PRESERVED, 14px patches, 2x2 merge.
    # Patch counts use CEIL, exactly as PixtralImageProcessorFast does ((d-1)//14 + 1).
    toks=[patch_tokens(w,h)[0] for w,h in sizes]
    print(f"  est. vision tokens: min {min(toks)}  med {q(toks,.5)}  p95 {q(toks,.95)}  max {max(toks)}")
    print(f"     (DeepSeek-OCR spent 273 or 1183 on these same images -- no tiling here)")

LE, PATCH, MERGE = CFG["longest_edge"], 14, 2
def patch_tokens(w, h):
    """Reproduce the library's own arithmetic: returns (merged_tokens, patch_h, patch_w)."""
    ratio = max(h/LE, w/LE)
    if ratio > 1: h, w = int(h//ratio), int(w//ratio)
    hp, wp = (h-1)//PATCH + 1, (w-1)//PATCH + 1     # ceil, as the processor does
    return (hp//MERGE) * (wp//MERGE), hp, wp

stats(TRAIN_ALL,"TRAIN (all)"); stats(TEST,"TEST")

# ---- pre-flight: crops too small for the 2x2 patch merger --------------------
# LightOnOcrPatchMerger.forward runs F.unfold(kernel_size=2, stride=2) on the patch grid.
# A crop whose short side rounds to a SINGLE 14px patch yields zero merged tokens and raises
# "Kernel size can't be greater than actual input size" inside the model forward. It is not
# caught earlier: with 0 image tokens the processor emits an empty replacement and
# _check_special_mm_tokens compares 0 == 0 and passes. One bad crop drawn at epoch 4 kills
# the run hours in, so check all 1135 here -- it costs seconds.
bad, at_risk = [], []
for r in TRAIN_ALL + TEST:
    w, h = Image.open(r["image"]).size
    t, hp, wp = patch_tokens(w, h)
    if hp < MERGE or wp < MERGE: bad.append((r["id"], w, h, hp, wp))
    ta, hpa, wpa = patch_tokens(max(1,int(w*0.93)), max(1,int(h*0.93)))   # augment can shrink 7%
    if hpa < MERGE or wpa < MERGE: at_risk.append((r["id"], w, h))
over_cap = [(r["id"], patch_tokens(*Image.open(r["image"]).size)[0])
            for r in TRAIN_ALL + TEST
            if patch_tokens(*Image.open(r["image"]).size)[0] > CFG["max_patches_per_image"]]
print(f"\nimages above the {CFG['max_patches_per_image']}-patch cap (Cell 5 downscales these): "
      f"{len(over_cap)}/{len(TRAIN_ALL)+len(TEST)}  {sorted(over_cap, key=lambda t:-t[1])[:4]}")
print(f"crops that would crash the patch merger as-is      : {len(bad)}")
print(f"crops that would crash after augmentation (x0.93) : {len(at_risk)}  {at_risk[:5]}")
print(f"=> Cell 5 upscales any crop whose short side is < {MIN_SHORT_SIDE}px, which covers both.")
if bad:
    print(f"   still-bad after the guard would be: {bad[:5]}")
print("\nexample:", TRAIN_ALL[0])


dataset root: /kaggle/input/datasets/uom190232v/handwritten-data/handwritten-data

Integrity check vs IJDAR Table 2 (SinOCR-Handwritten):
  train: images   908 (paper   908)   code points  17122 (paper  17122)   MATCH
  test : images   227 (paper   227)   code points   4029 (paper   4029)   MATCH
  => This IS the published SinOCR-Handwritten split; CER is comparable to 0.5253.

TRAIN (all) (n=908)
  chars/sample : min 3  med 15  p95 46  max 70
  width  px    : min 48  med 427  p95 3536  max 9248
  height px    : min 16  med 76  p95 491  max 1609
  aspect ratio : min 0.9  med 5.7  max 21.8
  est. vision tokens: min 2  med 38  p95 440  max 770
     (DeepSeek-OCR spent 273 or 1183 on these same images -- no tiling here)

TEST (n=227)
  chars/sample : min 3  med 13  p95 47  max 58
  width  px    : min 62  med 363  p95 3360  max 7582
  height px    : min 24  med 76  p95 453  max 1011
  aspect ratio : min 1.1  med 5.0  max 16.9
  est. vision tokens: min 2  med 34  p95 416  max 605
     (Deep

In [8]:
# ============================================================================
# Cell 3 — Train/val split (val from TRAIN) + seen/unseen-text tagging
# ============================================================================
import random
rng = random.Random(CFG["seed"])

def bucket(t):
    n=len(t); return 0 if n<10 else 1 if n<20 else 2 if n<35 else 3
by={}
for r in TRAIN_ALL: by.setdefault(bucket(r["text"]),[]).append(r)

TRAIN, VAL = [], []
for b, rows in sorted(by.items()):
    rows=rows[:]; rng.shuffle(rows)
    k=max(1,int(round(CFG["val_fraction"]*len(rows))))
    VAL.extend(rows[:k]); TRAIN.extend(rows[k:])
rng.shuffle(TRAIN); rng.shuffle(VAL)

assert not ({r["id"] for r in TRAIN} & {r["id"] for r in VAL}), "train/val overlap"
assert not ({r["id"] for r in TRAIN+VAL} & {r["id"] for r in TEST}), "TRAIN/VAL LEAKED INTO TEST"
print(f"train {len(TRAIN)}   val {len(VAL)}   test {len(TEST)}  (val carved from train)")
print("assert passed: no train/val id appears in test")

train_used = {r["text"].strip() for r in TRAIN}
train_all  = {r["text"].strip() for r in TRAIN_ALL}
for r in TEST:
    r["seen_in_train_used"] = r["text"].strip() in train_used
    r["seen_in_train_all"]  = r["text"].strip() in train_all
print(f"\ntest rows whose text occurs in what we train on : "
      f"{sum(r['seen_in_train_used'] for r in TEST)}/{len(TEST)}")
print(f"test rows whose text occurs in all 908 train rows: "
      f"{sum(r['seen_in_train_all'] for r in TEST)}/{len(TEST)}  <- TrOCR's 0.5253 condition")

if CFG["smoke_test"]:
    TRAIN, VAL = TRAIN[:16], VAL[:6]
    print(f"\nSMOKE TEST: truncated to train {len(TRAIN)}, val {len(VAL)}")


train 817   val 91   test 227  (val carved from train)
assert passed: no train/val id appears in test

test rows whose text occurs in what we train on : 72/227
test rows whose text occurs in all 908 train rows: 77/227  <- TrOCR's 0.5253 condition


In [9]:
# ============================================================================
# Cell 4 — Load LightOnOCR in 4-bit + warm-start from their Sinhala PRINT adapter
# ============================================================================
import sys, torch

# Re-apply the torchao patch now that peft is definitely imported (see Cell 1).
import peft
_patched = []
for _n, _m in list(sys.modules.items()):
    if _n.startswith("peft") and _m is not None and hasattr(_m, "is_torchao_available"):
        _f = getattr(_m, "is_torchao_available")
        if hasattr(_f, "cache_clear"):
            try: _f.cache_clear()
            except Exception: pass
        setattr(_m, "is_torchao_available", lambda: False); _patched.append(_n)
from peft.tuners.lora.torchao import is_torchao_available as _chk
assert _chk() is False, "torchao patch did not take; get_peft_model() will raise"
print(f"torchao probe disabled in {len(_patched)} peft module(s); verified -> False")

from transformers import (LightOnOcrProcessor, LightOnOcrForConditionalGeneration,
                          BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, prepare_model_for_kbit_training
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

MODEL_ID = CFG["base_repo"]
processor = LightOnOcrProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"        # their Exp 7 setting
tok = processor.tokenizer
print(f"processor loaded | pad={tok.pad_token!r}({tok.pad_token_id}) eos={tok.eos_token!r}({tok.eos_token_id})")

quant_cfg = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = COMPUTE_DTYPE,   # fp16 on T4, not bf16
    bnb_4bit_use_double_quant = True,
) if CFG["load_in_4bit"] else None

# transformers 5 renamed `torch_dtype` -> `dtype`; their Exp 7 (on 5.0.0) still used the old
# name. Try the new one and fall back, so this works either way.
# NB: from_pretrained has a **kwargs tail, so a wrong kwarg name is SWALLOWED, never a
# TypeError -- a try/except fallback here would be dead code and could hide a silent fp32
# load. Assert the resulting dtype instead.
model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=COMPUTE_DTYPE, device_map="auto", quantization_config=quant_cfg)
assert model.dtype == COMPUTE_DTYPE, f"dtype kwarg ignored: got {model.dtype}"
print(f"{MODEL_ID} | 4bit={CFG['load_in_4bit']} | VRAM {torch.cuda.memory_allocated()/1024**3:.2f} GB")

if CFG["load_in_4bit"]:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=CFG["grad_checkpointing"])

# ---- which modules will LoRA actually hit? -------------------------------
# Pixtral (vision) and Qwen3 (decoder) use the SAME linear names (q_proj, gate_proj, ...),
# so their Exp 7 target list adapts BOTH towers. This census proves what really happened
# instead of assuming it.
import collections
suffix = collections.Counter()
by_tower = collections.Counter()
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Linear) or m.__class__.__name__.startswith("Linear4bit"):
        s = name.split(".")[-1]
        suffix[s] += 1
        if "vision" in name: by_tower[(s, "vision")] += 1
        else:                by_tower[(s, "language")] += 1
print("\nLinear suffixes present (target_modules must match these):")
for k, v in suffix.most_common(16):
    nv = by_tower.get((k,"vision"),0); nl = by_tower.get((k,"language"),0)
    print(f"  {k:14} x{v:<5}  (vision {nv}, language {nl})")

TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]  # their Exp 7
if CFG["warm_start"] and CFG["lora_r"] != 32:
    raise ValueError("warm_start=True requires lora_r=32 to match their adapter's shapes.")

model = get_peft_model(model, LoraConfig(
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    bias="none", task_type="CAUSAL_LM", target_modules=TARGETS,
))
n_mod = sum(1 for n,_ in model.named_modules() if n.endswith("lora_A"))
n_vis = sum(1 for n,_ in model.named_modules() if n.endswith("lora_A") and "vision" in n)
print(f"\nLoRA modules created: {n_mod} ({n_vis} vision, {n_mod-n_vis} language)  targets {TARGETS}")
print(f"  their adapter file contains 364 (168 vision + 196 language) -- verified from its"
      f" safetensors header, so these should match exactly when warm_start=True")
if n_mod == 0: raise RuntimeError("target_modules matched nothing.")
if CFG["warm_start"] and n_mod != 364:
    raise RuntimeError(f"expected 364 LoRA modules to match their adapter, got {n_mod}; "
                       f"the warm start would be partial.")

# ------------------------------------------------- warm start (stage 1) ----
WARM_START_OK = False
if CFG["warm_start"]:
    sd = load_file(hf_hub_download(CFG["print_adapter_repo"], "adapter_model.safetensors"))
    n_file = sum(1 for k in sd if "lora_" in k)
    res = set_peft_model_state_dict(model, sd, adapter_name="default")
    unexpected = [k for k in res.unexpected_keys if "lora_" in k]
    print(f"\nwarm start from {CFG['print_adapter_repo']} (their Experiment 7)")
    print(f"  LoRA tensors in file          : {n_file}")
    print(f"  tensors that could NOT be placed: {len(unexpected)}")
    if unexpected:
        print("  first unplaced:", unexpected[:5])
        raise RuntimeError("Warm start FAILED: shapes do not fit. Set CFG['warm_start']=False.")
    nz = sum(1 for n,p in model.named_parameters()
             if "lora_B" in n and torch.count_nonzero(p).item() > 0)
    tot = sum(1 for n,_ in model.named_parameters() if "lora_B" in n)
    print(f"  non-zero lora_B after load    : {nz}/{tot}")
    # lora_B is zero-initialised, so a non-zero one proves real weights arrived. Requiring
    # ALL of them also catches a PARTIAL warm start (e.g. vision left at random init).
    if nz != tot:
        raise RuntimeError(f"Warm start is partial: only {nz}/{tot} lora_B tensors were "
                           f"populated. Training a half-warm-started model would be invalid.")
    WARM_START_OK = True
    print("  => the adapter now carries Sinhala learned from 707 printed Act pages (CER 0.0105).")
else:
    print("\nwarm_start disabled: fresh LoRA (ablation). Expect much worse -- the base model")
    print("has no Sinhala and 17,122 training characters is very little to teach a script.")

if CFG["grad_checkpointing"]:
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        print("\ngradient checkpointing: ON (non-reentrant)")
    except Exception as e:
        print(f"\n[warn] gradient checkpointing unavailable ({e}); continuing (more VRAM).")
    try: model.enable_input_require_grads()
    except Exception as e: print(f"[warn] enable_input_require_grads failed: {e}")

    # gradient_checkpointing_enable() only reaches submodules that expose the flag. A real
    # T4 run OOM'd in BACKWARD at 24 x 311 MB = 7.28 GB of retained vision attention, which
    # is exactly what "the vision tower is not checkpointed" looks like. Measure it, then
    # try to switch it on explicitly rather than assuming.
    def _gc_on():
        return [n for n, m in model.named_modules() if getattr(m, "gradient_checkpointing", False)]
    on = _gc_on(); vis = [n for n in on if "vision" in n]
    print(f"  checkpointing active on {len(on)} module(s); {len(vis)} in the vision tower")
    if not vis:
        forced = 0
        for n, m in model.named_modules():
            if "vision" in n and hasattr(m, "gradient_checkpointing"):
                m.gradient_checkpointing = True; forced += 1
        on = _gc_on(); vis = [n for n in on if "vision" in n]
        print(f"  vision tower was NOT checkpointed; set the flag on {forced} module(s) "
              f"-> now {len(vis)} active")
        if not vis:
            print("  [warn] the vision tower still reports no checkpointing. The "
                  f"{CFG['max_patches_per_image']}-patch cap is what keeps memory in budget.")

tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
to = sum(p.numel() for p in model.parameters())
print(f"\ntrainable params: {tr:,} / {to:,} ({100*tr/to:.3f}%)")
print(f"  (DeepSeek track had 86,184,960 trainable on the same 817 samples and overfit.)")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


torchao probe disabled in 4 peft module(s); verified -> False


You are using a model of type mistral3 to instantiate a model of type lighton_ocr. This is not supported for all configurations of models and can yield errors.


processor loaded | pad='<|endoftext|>'(151643) eos='<|im_end|>'(151645)


Loading weights:   0%|          | 0/532 [00:00<?, ?it/s]

lightonai/LightOnOCR-2-1B | 4bit=True | VRAM 1.70 GB

Linear suffixes present (target_modules must match these):
  gate_proj      x52     (vision 24, language 28)
  up_proj        x52     (vision 24, language 28)
  down_proj      x52     (vision 24, language 28)
  k_proj         x52     (vision 24, language 28)
  v_proj         x52     (vision 24, language 28)
  q_proj         x52     (vision 24, language 28)
  o_proj         x52     (vision 24, language 28)
  merging_layer  x1      (vision 1, language 0)
  linear_1       x1      (vision 1, language 0)
  linear_2       x1      (vision 1, language 0)
  lm_head        x1      (vision 0, language 1)

LoRA modules created: 364 (168 vision, 196 language)  targets ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  their adapter file contains 364 (168 vision + 196 language) -- verified from its safetensors header, so these should match exactly when warm_start=True

warm start from avishadilhara/sinhala-lightonocr-

In [10]:
# ============================================================================
# Cell 5 — Collator (their Exp 7 collator) + handwriting augmentation
# ============================================================================
import torch, random as _random
from PIL import Image, ImageOps, ImageFilter, ImageEnhance

LONGEST_EDGE = CFG["longest_edge"]
MAX_LENGTH   = CFG["max_length"]

# Their Exp 7 hardcodes ASSISTANT_START_PATTERN = [151645, 198, 151644, 77091, 198].
# Derive it from the tokenizer instead, then check it against their constant, so a chat
# template change fails loudly here rather than silently masking every label.
ASSISTANT_MARKER = "<|im_end|>\n<|im_start|>assistant\n"
ASSISTANT_START_PATTERN = processor.tokenizer.encode(ASSISTANT_MARKER, add_special_tokens=False)
THEIR_PATTERN = [151645, 198, 151644, 77091, 198]
print(f"assistant marker {ASSISTANT_MARKER!r}")
print(f"  derived : {ASSISTANT_START_PATTERN}")
print(f"  theirs  : {THEIR_PATTERN}   match={ASSISTANT_START_PATTERN == THEIR_PATTERN}")
if ASSISTANT_START_PATTERN != THEIR_PATTERN:
    print("  [warn] differs from their constant -- using the derived one (it matches THIS tokenizer).")


def _bg_level(img):
    g=img.convert("L"); w,h=g.size
    px=list(g.crop((0,0,w,1)).getdata())+list(g.crop((0,h-1,w,h)).getdata()) \
      +list(g.crop((0,0,1,h)).getdata())+list(g.crop((w-1,0,w,h)).getdata())
    px.sort(); return int(px[len(px)//2]) if px else 255

def cap_patches(img, max_patches=None, le=None, patch=14):
    """Downscale an image whose post-rescale patch grid would exceed the cap.

    The Pixtral tower's attention is quadratic in patch count and (measured) is not
    gradient-checkpointed here, so 24 layers of a 3,190-patch image retain ~7.3 GB. Capping
    per image leaves the median (168 patches) and 98.8% of the dataset untouched, which is
    far less destructive than lowering `longest_edge` globally -- that was their headline
    finding (0.1413 -> 0.0105) and should not be traded away for the sake of ~1% of crops."""
    max_patches = max_patches or CFG["max_patches_per_image"]
    le = le or LONGEST_EDGE
    w, h = img.size
    ratio = max(h/le, w/le)
    w2, h2 = (int(w//ratio), int(h//ratio)) if ratio > 1 else (w, h)
    n = ((h2-1)//patch + 1) * ((w2-1)//patch + 1)
    if n <= max_patches:
        return img
    sc = (max_patches / n) ** 0.5
    return img.resize((max(2*patch, int(w2*sc)), max(2*patch, int(h2*sc))), Image.LANCZOS)


def ensure_min_side(img, min_side=MIN_SHORT_SIDE):
    """Upscale so the short side clears the 2x2 patch merger's hard floor (15px).
    Applied AFTER augmentation, because the rescale step can shrink a crop by 7%:
    image_457 is 88x16px and augmentation takes it to 14px -> 1 patch -> RuntimeError."""
    w, h = img.size
    if min(w, h) >= min_side: return img
    s = min_side / min(w, h)
    return img.resize((max(min_side, round(w*s)), max(min_side, round(h*s))), Image.LANCZOS)


def augment_crop(img, rnd):
    """Conservative augmentation. Sinhala diacritics are tiny; aggressive blur or erosion
    destroys exactly the marks that distinguish characters."""
    bg=_bg_level(img); fill=(bg,bg,bg) if img.mode=="RGB" else bg
    if rnd.random()<0.7:
        img=img.rotate(rnd.uniform(-1.2,1.2), resample=Image.BILINEAR, expand=True, fillcolor=fill)
    if rnd.random()<0.4:
        s=rnd.uniform(-0.04,0.04); w,h=img.size
        img=img.transform((w+int(abs(s)*h),h), Image.AFFINE,
                          (1,s,-s*h if s<0 else 0,0,1,0), resample=Image.BILINEAR, fillcolor=fill)
    if rnd.random()<0.5:
        f=rnd.uniform(0.93,1.07)
        img=img.resize((max(8,int(img.width*f)), max(8,int(img.height*f))), Image.LANCZOS)
    if rnd.random()<0.6: img=ImageEnhance.Brightness(img).enhance(rnd.uniform(0.90,1.10))
    if rnd.random()<0.6: img=ImageEnhance.Contrast(img).enhance(rnd.uniform(0.88,1.14))
    if rnd.random()<0.25: img=img.filter(ImageFilter.GaussianBlur(rnd.uniform(0.3,0.7)))
    r=rnd.random()
    if   r<0.12: img=img.filter(ImageFilter.MinFilter(3))
    elif r<0.24: img=img.filter(ImageFilter.MaxFilter(3))
    return img


class Collator:
    """Their Exp 7 collate_fn, with three deliberate changes:
       (1) the assistant pattern is derived, not hardcoded (above);
       (2) pixel_values cast to COMPUTE_DTYPE (fp16 on T4) rather than hardcoded bf16;
       (3) optional augmentation on the raw crop.
    """
    def __init__(self, augment=False, seed=0):
        self.augment = augment
        self.seed = seed
        self._wid, self._rnd = None, _random.Random(seed)

    def _rng(self):
        """DataLoader workers are forked and inherit an identical RNG state, so a single
        shared Random() makes every worker draw the SAME augmentations. Re-seed per worker."""
        import torch.utils.data as _tud
        wi = _tud.get_worker_info()
        wid = wi.id if wi is not None else -1
        if self._wid != wid:
            self._wid, self._rnd = wid, _random.Random(self.seed * 1000 + wid + 1)
        return self._rnd

    def __call__(self, examples):
        images, msgs = [], []
        for ex in examples:
            img = Image.open(ex["image"]).convert("RGB") if isinstance(ex["image"], str) \
                  else ex["image"].convert("RGB")
            if self.augment: img = augment_crop(img, self._rng())
            img = ensure_min_side(cap_patches(img))
            images.append(img)
            msgs.append([
                {"role":"user",      "content":[{"type":"image"}]},          # no text prompt
                {"role":"assistant", "content":[{"type":"text","text":ex["text"].strip()}]},
            ])
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                 for m in msgs]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_LENGTH,
                           size={"longest_edge": LONGEST_EDGE})

        labels = inputs["input_ids"].clone()
        pad_id = processor.tokenizer.pad_token_id
        n_valid = []
        P = ASSISTANT_START_PATTERN
        for i in range(len(labels)):
            ids = inputs["input_ids"][i].tolist()
            start = None
            for j in range(len(ids)-len(P)):
                if ids[j:j+len(P)] == P: start = j+len(P); break
            labels[i,:] = -100
            if start is None:
                n_valid.append(0); continue
            c=0
            for j in range(start, len(ids)):
                if ids[j] == pad_id: break
                labels[i,j] = inputs["input_ids"][i,j]; c+=1
            n_valid.append(c)
        labels[inputs["input_ids"] == pad_id] = -100
        inputs["labels"] = labels
        inputs["pixel_values"] = inputs["pixel_values"].to(COMPUTE_DTYPE)
        if sum(n_valid) == 0:
            raise ValueError("Batch has 0 supervised label tokens -- assistant marker not found.")
        return inputs

train_collator = Collator(augment=CFG["augment"], seed=CFG["seed"])
eval_collator  = Collator(augment=False, seed=CFG["seed"])
print(f"\ncollators ready | longest_edge={LONGEST_EDGE} | max_length={MAX_LENGTH} "
      f"| dtype={COMPUTE_DTYPE} | train augment={CFG['augment']}")


assistant marker '<|im_end|>\n<|im_start|>assistant\n'
  derived : [151645, 198, 151644, 77091, 198]
  theirs  : [151645, 198, 151644, 77091, 198]   match=True

collators ready | longest_edge=1540 | max_length=4096 | dtype=torch.bfloat16 | train augment=True


In [11]:
# ============================================================================
# Cell 6 — Datasets, measured lengths, worst-case forward/backward smoke test
# ============================================================================
import torch
from torch.utils.data import Dataset

class OCRDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

train_ds, val_ds = OCRDataset(TRAIN), OCRDataset(VAL)
print(f"datasets: train {len(train_ds)}  val {len(val_ds)}")

# Derived from TRAINING data only -- a generation hyperparameter must not be tuned on test.
tgt = sorted(len(processor.tokenizer.encode(r["text"], add_special_tokens=False))
             for r in TRAIN_ALL)
q = lambda v,p: v[min(len(v)-1,int(p*len(v)))]
MAX_TARGET_TOKENS = tgt[-1]
print(f"\ntarget length in tokens (908 TRAIN rows): min {tgt[0]}  med {q(tgt,.5)}  "
      f"p95 {q(tgt,.95)}  max {MAX_TARGET_TOKENS}")
print(f"  chars/token = {sum(len(r['text']) for r in TRAIN_ALL)/sum(tgt):.2f}  "
      f"(a value near 1.0 means near-byte-level Sinhala tokenisation -- slower to learn)")

EVAL_MAX_NEW_TOKENS = CFG["eval_max_new_tokens"] or int(2*MAX_TARGET_TOKENS + 16)
print(f"EVAL_MAX_NEW_TOKENS = {EVAL_MAX_NEW_TOKENS}   "
      f"(DEVIATION: their Exp 7 used 2048; our targets are <= {MAX_TARGET_TOKENS} tokens)")

# Worst case = the most PATCHES, which is what drives memory -- not the most pixels.
# (A 8718x1228 crop scales to 1540x217 = 1760 patches, while a squarer 3464x896 crop
#  scales to 1540x398 = 3190 patches. Ranking by pixel area picks the wrong one.)
from PIL import Image as _PIL
ranked = []
for i, r in enumerate(TRAIN):
    w, h = _PIL.open(r["image"]).size
    t, hp, wp = patch_tokens(w, h)
    ranked.append((hp*wp, i, w, h, hp, wp))
ranked.sort(reverse=True)
worst = ranked[:CFG["per_device_batch"]]
print("\n--- forward/backward smoke test (worst-case samples, by patch count) ---")
for npatch,i,w,h,hp,wp in worst:
    print(f"  {TRAIN[i]['id']}: {w}x{h}px -> grid {hp}x{wp} = {npatch} patches")
tot_p = min(sum(x[0] for x in worst),
            CFG["per_device_batch"] * CFG["max_patches_per_image"])   # after Cell 5's cap
print(f"  batch total (after the {CFG['max_patches_per_image']}-patch cap) ~{tot_p} patches")
print(f"  -> one attention matrix ~{16*(tot_p**2)*2/1024**3:.2f} GB; if the vision tower is NOT")
print(f"     checkpointed, x24 layers = ~{24*16*(tot_p**2)*2/1024**3:.2f} GB retained in backward")
_vis_gc = [n for n, m in model.named_modules()
           if "vision" in n and getattr(m, "gradient_checkpointing", False)]
print(f"  vision-tower modules with checkpointing active: {len(_vis_gc)}")

batch = train_collator([train_ds[i] for _,i,_,_,_,_ in worst])
print("batch:", {k:(tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k,v in batch.items()})
print(f"  sequence length: {batch['input_ids'].shape[1]}  (cap {MAX_LENGTH})")
n_sup = int((batch["labels"] != -100).sum())
print(f"  supervised label positions: {n_sup}")
assert n_sup > 0, "every label masked -- the assistant marker was not found"

dev = next(p.device for p in model.parameters() if p.requires_grad)
model.train()
mb = {k:(v.to(dev) if torch.is_tensor(v) else v) for k,v in batch.items()}
# Scale the loss the way Trainer will (fp16=True uses a GradScaler). An unscaled fp16
# backward can underflow the earliest layers' gradients to exactly zero, which would make
# the vision-tower gradient count below read 0 and look like a target-list bug.
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16_AMP)
try:
    with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        out = model(**mb)
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError(
        f"OOM with per_device_batch={CFG['per_device_batch']} on a {tot_p}-patch batch "
        f"(T4 has ~14.6 GB usable).\n{e}\n\n"
        f"Pixtral attends over the CONCATENATED patches of a batch and pads every image to the\n"
        f"batch's max height, so cost grows with the SQUARE of the batch's total patch count.\n"
        f"Remedies, in order:\n"
        f"  1. Lower CFG['max_patches_per_image'] (now {CFG['max_patches_per_image']}) to 1600,\n"
        f"     then 1200. This touches only the largest ~12-14% of crops and leaves\n"
        f"     longest_edge=1540 intact for the rest.\n"
        f"  2. CFG['per_device_batch'] is already 1; raise CFG['grad_accum'] instead if you\n"
        f"     want a larger effective batch.\n"
        f"  3. Only then CFG['longest_edge']=1024 (their Exp 6 value) -- a DEVIATION that must\n"
        f"     be reported, since resolution was their key differentiator (0.1413 -> 0.0105).\n"
        f"  4. CFG['force_dtype']='fp16' is NOT an option -- it NaN'd on this model (rule 2b)."
    ) from e
loss = out.loss if hasattr(out,"loss") else out["loss"]
print(f"loss = {loss.item():.4f}")
if not torch.isfinite(loss):
    # Localise the NaN instead of just failing, so one Kaggle run tells us where it starts.
    print("\n### NaN/Inf LOCALISATION ###")
    print(f"  pixel_values finite : {torch.isfinite(mb['pixel_values']).all().item()}"
          f"  (absmax {mb['pixel_values'].abs().max().item():.3e})")
    lg = getattr(out, "logits", None)
    if lg is not None:
        print(f"  logits finite       : {torch.isfinite(lg).all().item()}"
              f"  (absmax {lg.abs().max().item():.3e})")
        print(f"  logits dtype        : {lg.dtype}")
    n_sup2 = int((mb["labels"] != -100).sum())
    print(f"  supervised labels   : {n_sup2}  (0 here would make the loss 0/0 = nan)")
    print(f"  COMPUTE_DTYPE       : {COMPUTE_DTYPE}")
    raise AssertionError(
        f"loss is NaN/Inf on the first batch with COMPUTE_DTYPE={COMPUTE_DTYPE}.\n"
        f"Remedies, in order:\n"
        f"  1. CFG['force_dtype'] = 'bf16'  -- this model is bf16 at every config level and\n"
        f"     their Exp 7 ran bf16; fp16's ~65504 ceiling overflows in Pixtral/Qwen3.\n"
        f"  2. CFG['force_dtype'] = 'fp32'  -- numerically safest; slower and more VRAM,\n"
        f"     but the smoke test only used ~1.2 GB of 14.6 GB so there is room.\n"
        f"  3. If logits are finite but the loss is not, the problem is label masking,\n"
        f"     not precision -- check the 'supervised labels' count above.")
scaler.scale(loss).backward()
g = [(n, float(p.grad.abs().mean())) for n,p in model.named_parameters()
     if p.requires_grad and p.grad is not None and p.grad.abs().sum() > 0]
print(f"parameters with non-zero gradient: {len(g)}")
assert len(g) > 0, "no gradients reached the LoRA parameters"
nv = sum(1 for n,_ in g if "vision" in n)
print(f"  of which in the VISION tower: {nv}  (Pixtral shares q_proj/gate_proj names with Qwen3,")
print(f"   so their Exp 7 target list adapts both towers -- confirmed here, not assumed)")
model.zero_grad(set_to_none=True)
print(f"peak VRAM (smoke test): {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
torch.cuda.reset_peak_memory_stats()   # so Cell 9 reports TRAINING peak, not this
del out, loss, mb; gc.collect(); torch.cuda.empty_cache()
print("smoke test PASSED")


datasets: train 817  val 91

target length in tokens (908 TRAIN rows): min 3  med 22  p95 69  max 105
  chars/token = 0.69  (a value near 1.0 means near-byte-level Sinhala tokenisation -- slower to learn)
EVAL_MAX_NEW_TOKENS = 226   (DEVIATION: their Exp 7 used 2048; our targets are <= 105 tokens)

--- forward/backward smoke test (worst-case samples, by patch count) ---
  image_1226: 3464x896px -> grid 29x110 = 3190 patches
  batch total (after the 2400-patch cap) ~2400 patches
  -> one attention matrix ~0.17 GB; if the vision tower is NOT
     checkpointed, x24 layers = ~4.12 GB retained in backward
  vision-tower modules with checkpointing active: 25
batch: {'input_ids': (1, 613), 'attention_mask': (1, 613), 'pixel_values': (1, 3, 350, 1344), 'image_sizes': (1, 2), 'labels': (1, 613)}
  sequence length: 613  (cap 4096)
  supervised label positions: 25


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


loss = 2.5605
parameters with non-zero gradient: 728
  of which in the VISION tower: 336  (Pixtral shares q_proj/gate_proj names with Qwen3,
   so their Exp 7 target list adapts both towers -- confirmed here, not assumed)
peak VRAM (smoke test): 3.51 GB
smoke test PASSED


In [12]:
# ============================================================================
# Cell 7 — Inference (train/eval parity) + the metric module
# ============================================================================
import torch, re, contextlib

DEVICE = next(p.device for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def ocr_predict(row, max_new_tokens=None, use_adapter=True):
    """Same chat format as training, with add_generation_prompt=True."""
    img = Image.open(row["image"]).convert("RGB") if isinstance(row, dict) else row
    img = ensure_min_side(cap_patches(img))   # same caps as training (see Cell 5)
    text = processor.apply_chat_template(
        [{"role":"user","content":[{"type":"image"}]}], tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, images=[img], return_tensors="pt",
                       size={"longest_edge": LONGEST_EDGE}).to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]
    inputs["pixel_values"] = inputs["pixel_values"].to(COMPUTE_DTYPE)
    model.eval()
    ctx = model.disable_adapter() if (not use_adapter and hasattr(model,"disable_adapter")) \
          else contextlib.nullcontext()
    with ctx, torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens or EVAL_MAX_NEW_TOKENS,
                             do_sample=False)
    # Decode ONLY the new tokens. Their testing notebook decodes the full sequence and then
    # deletes the words "system"/"user"/"assistant" from it; that also deletes any genuine
    # occurrence, which would flatter the base-model probe in Cell 8 (it emits English).
    # We have the exact prompt boundary, so slice instead.
    out = processor.batch_decode(ids[:, prompt_len:], skip_special_tokens=True)[0]
    return out

def normalize_text(t):
    """Applied to BOTH reference and hypothesis. Whitespace only -- never touches Sinhala.
    Note U+200D (ZWJ) is real Sinhala content (conjuncts) and is deliberately preserved;
    only U+200B (zero-width SPACE) is removed. Measured on this dataset: 5 of 1135
    references change (4 double-spaces, 1 leading space), 0 contain U+200B."""
    if t is None: return ""
    t = t.replace("\u200b", "")
    return re.sub(r"\s+", " ", t).strip()

def strip_scaffolding(t):
    """Hypothesis only: removes wrappers the model may emit around the answer."""
    if t is None: return ""
    t = re.sub(r"<\|[^>]*\|>", "", t)
    t = re.sub(r"^\s*```[a-zA-Z]*|```\s*$", "", t)
    return t

def postprocess(t):
    return normalize_text(strip_scaffolding(t))

def edit_counts(ref, hyp):
    """Levenshtein with an operation breakdown. N = len(ref) in code points (IJDAR Eq. 1)."""
    r,h=list(ref),list(hyp); n,m=len(r),len(h)
    d=[[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): d[i][0]=i
    for j in range(m+1): d[0][j]=j
    for i in range(1,n+1):
        for j in range(1,m+1):
            d[i][j]=min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+(r[i-1]!=h[j-1]))
    S=D=I=C=0; i,j=n,m
    while i>0 or j>0:
        if i>0 and j>0 and r[i-1]==h[j-1] and d[i][j]==d[i-1][j-1]: C+=1;i-=1;j-=1
        elif i>0 and j>0 and d[i][j]==d[i-1][j-1]+1:                S+=1;i-=1;j-=1
        elif i>0 and d[i][j]==d[i-1][j]+1:                          D+=1;i-=1
        else:                                                       I+=1;j-=1
    return S,D,I,C,n

def word_edit(ref,hyp):
    r,h=ref.split(),hyp.split(); n,m=len(r),len(h)
    d=[[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): d[i][0]=i
    for j in range(m+1): d[0][j]=j
    for i in range(1,n+1):
        for j in range(1,m+1):
            d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+(r[i-1]!=h[j-1]))
    return d[n][m], n

def score(pairs):
    """Three CER variants, because the two papers aggregate differently:
       cer_corpus     sum(S+D+I)/sum(N)        -- IJDAR Eq. 1 read literally
       cer_macro      mean of per-sample CER   -- the other reading of IJDAR
       cer_macro_clip mean of min(CER, 1.0)    -- what Cross-Temporal actually computes
                      (their testing notebook: metrics['CER'] = min(raw_cer, 1.0), then .mean())
       Reporting all three means the comparison holds whichever definition a reader assumes."""
    per, per_clip, e_t, n_t, we_t, wn_t, ex = [], [], 0, 0, 0, 0, 0
    for ref,hyp in pairs:
        S,D,I,C,N = edit_counts(ref,hyp); e=S+D+I
        c = e/N if N else (0.0 if not hyp else 1.0)
        per.append(c); per_clip.append(min(c,1.0))
        e_t+=e; n_t+=N
        we,wn = word_edit(ref,hyp); we_t+=we; wn_t+=wn
        ex += int(ref.strip()==hyp.strip())
    return {"n":len(pairs),
            "cer_corpus": e_t/n_t if n_t else float("nan"),
            "cer_macro": float(np.mean(per)) if per else float("nan"),
            "cer_macro_clipped": float(np.mean(per_clip)) if per_clip else float("nan"),
            "wer_corpus": we_t/wn_t if wn_t else float("nan"),
            "exact_match": ex/len(pairs) if pairs else float("nan"),
            "empty_preds": sum(1 for _,h in pairs if not h.strip()),
            "per_sample_cer": per}

def bootstrap_ci(pairs, n_boot=2000, seed=0, alpha=0.05):
    cnt=[(lambda t:(t[0]+t[1]+t[2], t[4]))(edit_counts(r,h)) for r,h in pairs]
    e=np.array([c[0] for c in cnt],float); nn=np.array([c[1] for c in cnt],float)
    rs=np.random.RandomState(seed); idx=np.arange(len(cnt)); vals=[]
    for _ in range(n_boot):
        s=rs.choice(idx,size=len(idx),replace=True); den=nn[s].sum()
        vals.append(e[s].sum()/den if den else np.nan)
    lo,hi=np.nanpercentile(vals,[100*alpha/2,100*(1-alpha/2)])
    return float(lo),float(hi)

def evaluate(rows, label, max_new_tokens=None, use_adapter=True, show=3, save_as=None):
    t0=time.time(); recs=[]
    for k,r in enumerate(rows):
        raw=ocr_predict(r, max_new_tokens=max_new_tokens, use_adapter=use_adapter)
        hyp=postprocess(raw)
        # Normalise the REFERENCE the same way as the hypothesis. Otherwise whitespace the
        # model was never trained to emit (it trains on text.strip()) becomes an unavoidable
        # deletion, putting a floor under CER that the harness itself created.
        recs.append({"id":r["id"],"reference":normalize_text(r["text"]),"prediction":hyp,
                     "raw":raw,"seen_in_train_all":r.get("seen_in_train_all")})
        if k<show:
            print(f"  [{k}] ref : {r['text']}")
            print(f"      pred: {hyp}")
    pairs=[(x["reference"],x["prediction"]) for x in recs]
    m=score(pairs); lo,hi=bootstrap_ci(pairs,seed=CFG["seed"])
    m["cer_ci95"]=[lo,hi]; m["seconds"]=time.time()-t0
    print(f"\n{label}: CER(corpus) {m['cer_corpus']:.4f} [95% CI {lo:.4f}-{hi:.4f}]   "
          f"CER(macro) {m['cer_macro']:.4f}   CER(macro,clipped) {m['cer_macro_clipped']:.4f}   "
          f"WER {m['wer_corpus']:.4f}   exact {m['exact_match']:.4f}   n={m['n']}   "
          f"{m['seconds']/60:.1f} min")
    if m["empty_preds"]:
        print(f"  [warn] {m['empty_preds']}/{m['n']} predictions EMPTY -- a CER pinned near 1.0 "
              f"with no variance is a pipeline bug, not a weak model.")
    if save_as:
        with open(os.path.join(RUN_DIR,save_as),"w",encoding="utf-8") as f:
            json.dump({"label":label,
                       "metrics":{k:v for k,v in m.items() if k!="per_sample_cer"},
                       "records":recs}, f, ensure_ascii=False, indent=2)
    return m, recs

# sanity-check our metric against jiwer
try:
    import jiwer
    _r,_h = "හෘද සෑත්කමක්","හෘද සෑත්"
    S,D,I,C,N = edit_counts(_r,_h)
    print(f"metric check: ours (S+D+I)/N = {(S+D+I)/N:.6f}   jiwer.cer = {jiwer.cer(_r,_h):.6f}")
except Exception as e:
    print(f"(jiwer cross-check skipped: {e})")


metric check: ours (S+D+I)/N = 0.333333   jiwer.cer = 0.333333


In [13]:
# ============================================================================
# Cell 8 — Zero-shot baselines: THE decisive probe, before any training
# ============================================================================
# (a) base LightOnOCR with the adapter disabled -> a model with no Sinhala at all
# (b) their Sinhala PRINT adapter, untouched    -> does page-level printed Sinhala transfer
#                                                  to handwritten word crops?
#
# (b) is the experiment that failed in the DeepSeek track: there, the print adapter scored
# 5.37 vs the base model's 5.07, i.e. it actively hurt. If (b) beats (a) clearly here, the
# warm start is doing its job and the architecture change is justified. If (b) is also worse
# than (a), stop and reconsider before spending hours on training.
ZS = {}
T_ZS_START = time.time()          # so Cell 9 can subtract this from the training budget
if CFG["run_zero_shot"]:
    rows = TEST[:CFG["zeroshot_n"]]
    print(f"Zero-shot on {len(rows)} test samples (max_new_tokens={CFG['zeroshot_max_new_tok']})\n")

    # Probe (a) only needs to establish that the un-adapted model is hopeless, and it never
    # emits EOS (no Sinhala), so every sample runs the full token budget. Use a subset.
    base_rows = TEST[:CFG["zeroshot_base_n"]]
    print(f"--- (a) base LightOnOCR-2-1B, LoRA disabled (n={len(base_rows)}) ---")
    ZS["base"] = evaluate(base_rows, "zero-shot base LightOnOCR",
                          max_new_tokens=CFG["zeroshot_max_new_tok"],
                          use_adapter=False, save_as="zeroshot_base.json")[0]

    if WARM_START_OK:
        print("\n--- (b) their Sinhala PRINT adapter (Exp 7), un-finetuned on handwriting ---")
        ZS["print_adapter"] = evaluate(rows, "zero-shot Sinhala print adapter",
                                       max_new_tokens=CFG["zeroshot_max_new_tok"],
                                       use_adapter=True, save_as="zeroshot_print_adapter.json")[0]
        a, b = ZS["base"]["cer_corpus"], ZS["print_adapter"]["cer_corpus"]
        print(f"\n>>> TRANSFER CHECK: base {a:.4f} vs print-adapter {b:.4f}")
        print(f"    {'print adapter HELPS -- warm start justified' if b < a else 'print adapter does NOT help -- same failure as the DeepSeek track'}")
        print(f"    (DeepSeek track for reference: base 5.0742 vs print adapter 5.3728)")
    with open(os.path.join(RUN_DIR,"zeroshot_summary.json"),"w") as f:
        json.dump({k:{kk:vv for kk,vv in v.items() if kk!="per_sample_cer"} for k,v in ZS.items()},
                  f, indent=2)
    gc.collect(); torch.cuda.empty_cache()
else:
    print("run_zero_shot=False -> skipping")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Zero-shot on 227 test samples (max_new_tokens=192)

--- (a) base LightOnOCR-2-1B, LoRA disabled (n=64) ---
  [0] ref : හෘද සැත්කමක්
      pred: $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$ $\text{ローケ}$
  [1] ref : රෙදි මැසීම
      pred: $\mathcal{A} \in \mathcal{A}$ $\mathcal{B} \in \mathcal{B}$ $\mathcal{C} \in \mathcal{C}$ $\mathcal{D} \in \mathcal{D}$ $\mathcal{E} \in \mathcal{E}$ $\mathcal{F} \in \mathcal{F}$ $\mathcal{G} \in \mathcal{G}$ $\mathcal{H} \in \mathcal{H}$ $\mathcal{I} \in \mathcal{I}$ $\mathcal{J} \in \mathcal{J}$ $\mathcal{K} \in \mathcal{K}$ $\mathcal{L} \in \mathcal{L}$ $\mathcal{M} \in \mathcal{
  [2] ref : මානසික
      pred: $\mathcal{A} \mathcal{B} \mathcal{C} \mathcal{D} \mathcal{E} \mathcal{F} \mathcal{G} \mathc

In [14]:
# ============================================================================
# Cell 9 — Train (validation-CER early stopping + wall-clock guard)
# ============================================================================
from transformers import Trainer, TrainingArguments, TrainerCallback

class ValCERCallback(TrainerCallback):
    """DEVIATION from their Exp 7, which selected on eval_loss. Teacher-forced loss and
    free-running CER decouple: falling loss with flat CER means the decoder is fitting
    tokens without producing usable text. We select on the metric we report."""
    def __init__(self, rows, every, n, patience):
        self.rows, self.every, self.n, self.patience = rows, every, n, patience
        self.best, self.bad, self.history = float("inf"), 0, []
    def _score(self, state):
        pairs=[]
        for k,r in enumerate(self.rows[:self.n]):
            hyp=postprocess(ocr_predict(r)); pairs.append((r["text"],hyp))
            if k<2:
                print(f"    val[{k}] ref : {r['text']}")
                print(f"            pred: {hyp}")
        m=score(pairs)
        self.history.append({"step":state.global_step,"cer":m["cer_corpus"],
                             "exact":m["exact_match"],"empty":m["empty_preds"]})
        print(f"  >> step {state.global_step}: val CER {m['cer_corpus']:.4f}  "
              f"exact {m['exact_match']:.3f}  empty {m['empty_preds']}/{m['n']}")
        return m["cer_corpus"]
    def on_step_end(self, args, state, control, **kw):
        if state.global_step>0 and state.global_step % self.every == 0:
            cer=self._score(state)
            if cer < self.best - 1e-4:
                self.best, self.bad = cer, 0
                kw["model"].save_pretrained(BEST_DIR)
                print(f"     new best -> saved to {BEST_DIR}")
            else:
                self.bad += 1
                print(f"     no improvement ({self.bad}/{self.patience}); best {self.best:.4f}")
                if self.bad >= self.patience:
                    print("     early stopping"); control.should_training_stop = True
            model.train()
        return control

class WallClockGuard(TrainerCallback):
    def __init__(self, hours): self.deadline=time.time()+hours*3600; self.hours=hours
    def on_step_end(self, args, state, control, **kw):
        if time.time() > self.deadline:
            print(f"\n[budget] {self.hours} h reached at step {state.global_step}; stopping.")
            control.should_training_stop = True
        return control

if CFG["do_train"]:
    spe = max(1, len(TRAIN)//(CFG["per_device_batch"]*CFG["grad_accum"]))
    print(f"~{spe} optimizer steps/epoch x {CFG['epochs']} epochs = ~{spe*CFG['epochs']} steps "
          f"(effective batch {CFG['per_device_batch']*CFG['grad_accum']})")
    LR = CFG["lr_warm_start"] if CFG["warm_start"] else CFG["lr"]
    print(f"learning rate: {LR}  ({'stage-2 rate, continuing a converged adapter' if CFG['warm_start'] else 'cold-start rate (their Exp 7)'})")

    val_cb = ValCERCallback(VAL, CFG["eval_every_steps"], CFG["val_eval_n"], CFG["early_stop_patience"])

    # Score the warm-started adapter BEFORE any training and make it the incumbent best, so
    # "do not train at all" can win. Otherwise val_cb.best starts at infinity and whatever
    # the model looks like at the first eval is saved unconditionally -- even if training
    # has already damaged the printed-Sinhala representation.
    print("\n--- baseline: validation CER of the warm-started adapter, before training ---")
    _S0 = type("S", (), {"global_step": 0})()
    val_cb.best = val_cb._score(_S0)
    model.save_pretrained(BEST_DIR)
    print(f"  baseline val CER {val_cb.best:.4f} saved as the incumbent best")
    model.train()

    # Make the budget honest: the guard only bounds trainer.train(), so subtract what the
    # zero-shot cell already spent and reserve time for the final test evaluation.
    _spent  = (time.time() - T_ZS_START)/3600
    _budget = max(0.25, CFG["train_hours_budget"] - _spent - 0.35)
    print(f"\nzero-shot + setup used {_spent:.2f} h; training budget set to {_budget:.2f} h "
          f"(0.35 h reserved for the final test eval)")
    args = TrainingArguments(
        output_dir                  = os.path.join(RUN_DIR,"trainer"),
        num_train_epochs            = CFG["epochs"],
        per_device_train_batch_size = CFG["per_device_batch"],
        gradient_accumulation_steps = CFG["grad_accum"],
        learning_rate               = LR,
        lr_scheduler_type           = CFG["lr_scheduler"],
        warmup_steps                = CFG["warmup_steps"],
        weight_decay                = CFG["weight_decay"],
        max_grad_norm               = CFG["max_grad_norm"],
        logging_steps               = 10,
        eval_strategy               = "no",     # our own generation-based validation
        save_strategy               = "no",     # the callback saves the best adapter
        fp16                        = USE_FP16_AMP,
        bf16                        = not USE_FP16_AMP,
        optim                       = CFG["optim"],
        gradient_checkpointing      = False,    # already enabled in Cell 4
        dataloader_pin_memory       = False,
        dataloader_num_workers      = 2,
        remove_unused_columns       = False,    # REQUIRED for vision inputs
        report_to                   = "none",
        seed                        = CFG["seed"],
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      data_collator=train_collator,
                      callbacks=[val_cb, WallClockGuard(_budget)])
    t0=time.time(); stats_=trainer.train()
    print(f"\ntrained in {(time.time()-t0)/60:.1f} min | {stats_.metrics}")
    print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    with open(os.path.join(RUN_DIR,"train_history.json"),"w") as f:
        json.dump({"val_history":val_cb.history,"best_val_cer":val_cb.best,
                   "log_history":trainer.state.log_history,
                   "config":{k:str(v) for k,v in CFG.items()}}, f, indent=2)
    if os.path.isdir(BEST_DIR) and val_cb.best < float("inf"):
        print(f"\nreloading best adapter (val CER {val_cb.best:.4f})")
        sd=load_file(os.path.join(BEST_DIR,"adapter_model.safetensors"))
        r=set_peft_model_state_dict(model, sd, adapter_name="default")
        bad=[k for k in r.unexpected_keys if "lora_" in k]
        assert not bad, f"could not restore best adapter: {bad[:3]}"
        print("best adapter restored")
    else:
        print("\n[warn] no best checkpoint saved; evaluating final weights.")
    gc.collect(); torch.cuda.empty_cache()
else:
    print("do_train=False -> skipping")


~204 optimizer steps/epoch x 20 epochs = ~4080 steps (effective batch 4)
learning rate: 2e-05  (stage-2 rate, continuing a converged adapter)

--- baseline: validation CER of the warm-started adapter, before training ---
    val[0] ref : දේශාන් ලක්ෂිත වික්‍රමසිංහ
            pred: අද්උාන් ලක්ෂිත නිකුළිකිටන්
    val[1] ref : ඒකනායක ආරච්චිලාගේ ආකාශ් තාරුක
            pred: শ্রীরূপনামন্তর ৭২৮শিক্ষিদানং ৭২ক্ষাণঞ্চ ৭১৪৮
  >> step 0: val CER 5.9077  exact 0.000  empty 1/32
  baseline val CER 5.9077 saved as the incumbent best

zero-shot + setup used 0.85 h; training budget set to 5.30 h (0.35 h reserved for the final test eval)


Step,Training Loss
10,2.094850
20,1.404444
30,1.214052
40,1.396688
50,0.964560
60,1.076281
70,1.068588
80,0.873616
90,1.011993
100,0.875375


    val[0] ref : දේශාන් ලක්ෂිත වික්‍රමසිංහ
            pred: දේශාන් ලක්ෂිත විතුමසිංහ
    val[1] ref : ඒකනායක ආරච්චිලාගේ ආකාශ් තාරුක
            pred: ඒකතායක ආරප්ච්ලාගේ ආකෘත් කාරක
  >> step 400: val CER 0.3561  exact 0.094  empty 0/32
     new best -> saved to /kaggle/working/lightonocr_hw_run/best_adapter
    val[0] ref : දේශාන් ලක්ෂිත වික්‍රමසිංහ
            pred: දේශාන් ලක්ෂිත වික්‍රමසිංහ
    val[1] ref : ඒකනායක ආරච්චිලාගේ ආකාශ් තාරුක
            pred: ඒකානායක ආරච්චිලාගේ ආකෘත් කාරුණ
  >> step 800: val CER 0.3247  exact 0.094  empty 0/32
     new best -> saved to /kaggle/working/lightonocr_hw_run/best_adapter
    val[0] ref : දේශාන් ලක්ෂිත වික්‍රමසිංහ
            pred: දේශාන් ලක්ෂිත වික්‍රමසිංහ
    val[1] ref : ඒකනායක ආරච්චිලාගේ ආකාශ් තාරුක
            pred: ඒකකායක ආරච්චිලාගේ ආකෘත් කාරුණ
  >> step 1200: val CER 0.3100  exact 0.156  empty 0/32
     new best -> saved to /kaggle/working/lightonocr_hw_run/best_adapter
    val[0] ref : දේශාන් ලක්ෂිත වික්‍රමසිංහ
            pred: දේශාන් ලක්

In [15]:
# ============================================================================
# Cell 10 — Final evaluation on the held-out test split (touched only here)
# ============================================================================
if CFG["run_final_test_eval"]:
    print(f"Evaluating on all {len(TEST)} test samples (first and only use of the test split)\n")
    m_ft, recs = evaluate(TEST, "FINE-TUNED LightOnOCR + QLoRA", show=6,
                          save_as="final_test_predictions.json")

    seen   = [(r["reference"],r["prediction"]) for r in recs if r["seen_in_train_all"]]
    unseen = [(r["reference"],r["prediction"]) for r in recs if not r["seen_in_train_all"]]
    m_seen, m_unseen = (score(seen) if seen else None), (score(unseen) if unseen else None)

    buckets={"1-9 chars":[],"10-19":[],"20-34":[],"35+":[]}
    for r in recs:
        n=len(r["reference"])
        buckets["1-9 chars" if n<10 else "10-19" if n<20 else "20-34" if n<35 else "35+"] \
            .append((r["reference"],r["prediction"]))

    IJDAR={"TrOCR (printed only)":0.9940,"Tesseract (pre-trained)":0.9493,
           "Google Vision API":0.7532,"Tesseract (printed+handwritten)":0.7204,
           "TrOCR (printed->handwritten)":0.5253}
    OURS_DEEPSEEK = 0.7059   # our own DeepSeek-OCR V1 + QLoRA run, same split/metric

    print("\n"+"="*80)
    print("RESULTS — SinOCR-Handwritten test (n=227), CER = (S+D+I)/N on code points")
    print("="*80)
    print(f"{'Model':46} {'CER':>8} {'source':>22}")
    print("-"*80)
    for k,v in sorted(IJDAR.items(), key=lambda x:-x[1]):
        print(f"{k:46} {v:8.4f} {'IJDAR Table 5':>22}")
    print(f"{'DeepSeek-OCR V1 + QLoRA (our track 1)':46} {OURS_DEEPSEEK:8.4f} {'our run':>22}")
    for k,lab in (("base","LightOnOCR zero-shot"),
                  ("print_adapter","LightOnOCR + Sinhala PRINT LoRA (zero-shot)")):
        if k in ZS: print(f"{lab:46} {ZS[k]['cer_corpus']:8.4f} {'this notebook':>22}")
    print(f"{'LightOnOCR + QLoRA (handwritten) [OURS]':46} {m_ft['cer_corpus']:8.4f} {'this notebook':>22}")
    print("-"*80)
    bar=IJDAR["TrOCR (printed->handwritten)"]; d=bar-m_ft["cer_corpus"]
    print(f"vs the TrOCR bar (0.5253): {'BEATS' if d>0 else 'does NOT beat'} it by {abs(d):.4f} CER "
          f"({abs(100*d/bar):.1f}% relative)")
    print(f"vs our DeepSeek track (0.7059): "
          f"{'better' if m_ft['cer_corpus']<OURS_DEEPSEEK else 'worse'} by "
          f"{abs(OURS_DEEPSEEK-m_ft['cer_corpus']):.4f}")
    lo,hi=m_ft["cer_ci95"]
    print(f"95% CI [{lo:.4f}, {hi:.4f}] -> improvement over 0.5253 is "
          f"{'SIGNIFICANT' if hi < bar else 'NOT clearly significant'} at 95%")

    print("\nSecondary metrics and breakdowns\n"+"-"*80)
    print(f"  CER macro-average                    : {m_ft['cer_macro']:.4f}")
    print(f"  CER macro, clipped at 1.0            : {m_ft['cer_macro_clipped']:.4f}"
          f"   <- Cross-Temporal's own definition")
    print(f"  WER (corpus)                         : {m_ft['wer_corpus']:.4f}")
    print(f"  exact-match rate                     : {m_ft['exact_match']:.4f}")
    print(f"  empty predictions                    : {m_ft['empty_preds']}/{m_ft['n']}")
    if m_seen and m_unseen:
        print(f"  CER seen-text   (n={m_seen['n']:3})              : {m_seen['cer_corpus']:.4f}")
        print(f"  CER unseen-text (n={m_unseen['n']:3})              : {m_unseen['cer_corpus']:.4f}"
              f"   <- honest estimate for new vocabulary")
    print("  by reference length:")
    for k,v in buckets.items():
        if v: print(f"    {k:12} n={len(v):3}  CER {score(v)['cer_corpus']:.4f}")
    print("  (the DeepSeek run was INVERTED here -- worse on short refs, 0.766 vs 0.635 --")
    print("   because it leaned on context. Check whether that inversion is gone.)")

    def _scer(r):
        S,D,I,C,N=edit_counts(r["reference"],r["prediction"]); return (S+D+I)/max(1,N)
    print("\nWorst 8 predictions (error analysis):")
    for r in sorted(recs,key=_scer,reverse=True)[:8]:
        print(f"  CER {_scer(r):.2f}  ref : {r['reference']}")
        print(f"            pred: {r['prediction']}")

    report={"dataset":{"name":"SinOCR-Handwritten","train_used":len(TRAIN),"val":len(VAL),
                       "test":len(TEST),"identity_verified_vs_IJDAR_table2":True},
            "metric":"CER=(S+D+I)/N, N=reference code points (IJDAR Eq.1); macro_clipped=Cross-Temporal's",
            "config":{k:str(v) for k,v in CFG.items()},
            "results":{"ours_finetuned":{k:v for k,v in m_ft.items() if k!="per_sample_cer"},
                       "zero_shot":{k:{kk:vv for kk,vv in v.items() if kk!="per_sample_cer"}
                                    for k,v in ZS.items()},
                       "ijdar_table5_baselines":IJDAR,
                       "our_deepseek_track":OURS_DEEPSEEK,
                       "seen_text":({k:v for k,v in m_seen.items() if k!="per_sample_cer"} if m_seen else None),
                       "unseen_text":({k:v for k,v in m_unseen.items() if k!="per_sample_cer"} if m_unseen else None),
                       "by_length":{k:score(v)["cer_corpus"] for k,v in buckets.items() if v}}}
    with open(os.path.join(RUN_DIR,"RESULTS.json"),"w",encoding="utf-8") as f:
        json.dump(report,f,ensure_ascii=False,indent=2)
    model.save_pretrained(os.path.join(RUN_DIR,"final_adapter"))
    print(f"\nsaved: {RUN_DIR}/RESULTS.json, final_test_predictions.json, final_adapter/, best_adapter/")
    print("Download the whole lightonocr_hw_run/ folder before the Kaggle session ends.")
else:
    print("run_final_test_eval=False -> skipping")


Evaluating on all 227 test samples (first and only use of the test split)

  [0] ref : හෘද සැත්කමක්
      pred: හෘද දාර්කම්කම්
  [1] ref : රෙදි මැසීම
      pred: රුද්‍රි පැසිමට
  [2] ref : මානසික
      pred: ඉන්පිය
  [3] ref : සුරසිංහ ආරච්චිගේ නිපුන් තේජාන් ෆොන්සේකා
      pred: සුරණියක අව්විද්‍යේ විශ්ව පේරාන් ෆොන්සේකා
  [4] ref : නො 304 , ශ්‍රී වික්‍රම රාජ සිංහ පාර , 3 වන කුරණ , මීගමුව
      pred: නො 304 , ඉ වික්‍රම රාජ සිංහ පාර රාජ පුරණු මීගමුසි
  [5] ref : විදුලි ඉංජිනේරු ii පංතිය
      pred: පිළිලි ඉංජිනේරු මි පංතිය

FINE-TUNED LightOnOCR + QLoRA: CER(corpus) 0.2636 [95% CI 0.2379-0.2917]   CER(macro) 0.2910   CER(macro,clipped) 0.2837   WER 0.6404   exact 0.1982   n=227   12.8 min

RESULTS — SinOCR-Handwritten test (n=227), CER = (S+D+I)/N on code points
Model                                               CER                 source
--------------------------------------------------------------------------------
TrOCR (printed only)                             0.9940          IJDAR 

## Reading the output, and what to change next

### Read these three things first, in order
1. **Cell 1** — `revision 2026-09-11.1` and `CUDA smoke test : OK`. If the revision differs you
   are running an old copy.
2. **Cell 8's `TRANSFER CHECK`** — this decides whether the whole approach is sound. If the
   Sinhala print adapter does **not** beat the bare base model, the warm start is failing the
   same way it did in the DeepSeek track, and no amount of training will rescue it. Stop and
   reconsider rather than spending six hours.
3. **Cell 6's estimated vision tokens** vs DeepSeek's 273/1183. This is the mechanism the whole
   architecture change rests on; if the numbers are not dramatically lower, something is wrong
   with the resolution config.

### Ablations — each is one config change and one thesis table row

| Ablation | Change | What it answers |
|---|---|---|
| No warm start | `warm_start=False` | How much of the result is their printed-Sinhala stage 1? |
| Lower resolution | `longest_edge=1024` (their Exp 6) | Reproduces their key finding: resolution was the differentiator (0.1413 → 0.0105) |
| Much lower | `longest_edge=700` (their Exp 5) | The other end of their resolution sweep |
| Smaller LoRA | `lora_r=16, lora_alpha=16` (needs `warm_start=False`) | Their Exp 5 config; capacity vs 817 samples |
| No augmentation | `augment=False` | Does augmentation help or blur diacritics away? |
| Their selection rule | select on `eval_loss` | Does CER-based checkpoint selection actually matter? |

### Honest caveats for the write-up
- These numbers are **unverified** until this notebook runs — it has never been executed, since
  this project has no local GPU.
- The warm start means our model has seen **707 pages of printed Sinhala** that the TrOCR
  baseline did not (TrOCR saw 90,000 printed *line crops* instead). Both are "printed
  pre-training", but they are not the same corpus — say so rather than implying parity.
- ~34% of test rows share their exact text with a train row. That is a property of the
  *published* split and TrOCR's 0.5253 was measured under identical conditions, so the
  comparison is fair — but the unseen-text column is the better estimate for new vocabulary.
- Test **images** are passed to the model only in Cells 8 and 10; test **reference strings**
  are also read in Cell 3 for the seen/unseen tagging. No generation hyperparameter is derived
  from test (the length cap comes from TRAIN only), and no checkpoint is selected on test.
